# Loading and Understanding the CAP Sleep Database

**Dataset:** [CAP Sleep Database v1.0.0 (PhysioNet)](https://physionet.org/content/capslpdb/1.0.0/)

This notebook walks through, step by step:

1. What is in the dataset
2. Downloading a few recordings (you do **not** need all of them)
3. Loading the polysomnography (PSG) signals from the `.edf` files
4. Parsing the sleep-stage / CAP annotation `.txt` files
5. Plotting a hypnogram and basic sleep statistics
6. Looking at CAP A-phases and computing an (approximate) **CAP rate**
7. Comparing CAP rate across disorder groups using the annotation files only (small downloads)

### Background: what is CAP?
**Cyclic Alternating Pattern (CAP)** is a periodic EEG activity seen in NREM sleep. It is made of repeating
cycles of an **A phase** (a burst of activation, 2-60 s) followed by a **B phase** (the return to background, 2-60 s).
A-phases are classified as **A1** (mostly slow, synchronized EEG), **A2** (mixed) and **A3** (mostly fast, desynchronized).
**CAP rate** = (time spent in CAP during NREM sleep) / (total NREM time) x 100%.

### Dataset overview (from the PhysioNet page)
- 108 whole-night polysomnographic recordings
- Includes healthy controls and several pathologies (e.g. NFLE, REM behavior disorder, periodic leg movements, insomnia, narcolepsy, sleep-disordered breathing, bruxism)
- Each recording has an `.edf` file (signals) and a `.txt` file (sleep stages and CAP annotations)
- File name prefixes tell you the group: `n` = healthy, `nfle`, `rbd`, `plm`, `ins`, `narco`, `sdb`, `brux`
- Recordings have **different channel sets and sampling rates**, so never assume channel names are identical across subjects

## 1. Setup

In [ ]:
# Run this once (remove the '#' if you need to install the packages)
# %pip install mne pandas numpy matplotlib scipy

In [ ]:
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from scipy.signal import welch

mne.set_log_level("ERROR")
plt.rcParams["figure.dpi"] = 110

BASE_URL = "https://physionet.org/files/capslpdb/1.0.0/"
DATA_DIR = Path("capslpdb_data")      # files are saved next to this notebook
DATA_DIR.mkdir(exist_ok=True)
EPOCH_SEC = 30                        # sleep stages are scored in 30 s epochs

## 2. Download helper
The EDF files can be large, so start with one or two subjects. The annotation `.txt` files are small.

In [ ]:
def download(fname, dest_dir=DATA_DIR):
    """Download one file from the CAP database (skips it if already downloaded)."""
    dest = Path(dest_dir) / fname
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    print(f"Downloading {fname} ...")
    req = urllib.request.Request(BASE_URL + fname, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(dest, "wb") as f:
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
    return dest


# Start with one healthy control (n1). Add e.g. "rbd1" or "nfle1" later to compare groups.
record = "n1"
edf_path = download(f"{record}.edf")
txt_path = download(f"{record}.txt")
print(edf_path, f"{edf_path.stat().st_size / 1e6:.0f} MB")
print(txt_path, f"{txt_path.stat().st_size / 1e3:.0f} KB")

## 3. Load the signals (EDF)
MNE reads the file lazily (`preload=False`), so this is fast and light on memory.

In [ ]:
raw = mne.io.read_raw_edf(edf_path, preload=False)
print(raw)
print()
print("Channels:", raw.ch_names)
print("Sampling rate (Hz):", raw.info["sfreq"])
print(f"Duration: {raw.times[-1] / 3600:.2f} hours")
print("Recording start:", raw.info["meas_date"])

> **Note:** In EDF files each channel can have its own sampling rate. MNE brings them all to the highest rate,
> so the number above is the highest rate in the file. If `read_raw_edf` errors on a file, try
> `mne.io.read_raw_edf(path, infer_types=False)` or the `pyedflib` package.

In [ ]:
# A tidy table of channels (names differ between subjects, so always check!)
pd.DataFrame({"channel": raw.ch_names, "type": raw.get_channel_types()})

### Pick an EEG channel
Names vary across recordings (e.g. `C4-A1`, `C4A1`, `C3-A2` ...). This helper picks a central EEG derivation if one exists.
**Always check the result.**

In [ ]:
def pick_eeg_channel(ch_names):
    preferred = ["C4A1", "C3A2", "F4C4", "F3C3", "FP2F4", "FP1F3"]
    norm = {re.sub(r"[^A-Z0-9]", "", c.upper()): c for c in ch_names}
    for p in preferred:
        if p in norm:
            return norm[p]
    return ch_names[0]   # fallback: just take the first channel


eeg_ch = pick_eeg_channel(raw.ch_names)
print("Using EEG channel:", eeg_ch)

## 4. Look at a raw 30-second epoch

In [ ]:
def plot_epoch(raw, epoch_idx, channels, epoch_sec=EPOCH_SEC):
    start, stop = epoch_idx * epoch_sec, (epoch_idx + 1) * epoch_sec
    data, times = raw.get_data(picks=channels, tmin=start, tmax=stop, return_times=True)
    fig, axes = plt.subplots(len(channels), 1, figsize=(11, 1.6 * len(channels)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, ch, y in zip(axes, channels, data):
        ax.plot(times, y * 1e6, lw=0.6)
        ax.set_ylabel(ch, rotation=0, ha="right", va="center")
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"Epoch {epoch_idx} ({start}-{stop} s)  [units shown in microvolts if the file stores volts]")
    plt.tight_layout()
    plt.show()


plot_epoch(raw, epoch_idx=300, channels=raw.ch_names[:4])

## 5. Parse the annotation file
First, look at the **raw text** so you know exactly what the file looks like.

In [ ]:
with open(txt_path, "r", errors="ignore") as f:
    lines = f.readlines()

print("Total lines:", len(lines))
print("".join(lines[:25]))

The annotation file is tab-separated with a header row and (roughly) these columns:

| Column | Meaning |
|---|---|
| Sleep Stage | `W`, `S1`-`S4`, `R` (REM), `MT` (movement time) |
| Position | Body position |
| Time | Clock time of the event (`hh.mm.ss` or `hh:mm:ss`) |
| Event | `SLEEP-S0..S4`, `SLEEP-REM`, `SLEEP-MT` (stage rows), `MCAP-A1/A2/A3` (CAP A-phases), plus other events |
| Duration | Seconds (30 for stage rows) |
| Location | Channel where the event was marked |

The parser below is written defensively. **If the printout above looks different, adjust it.**

In [ ]:
def _clock_to_sec(t):
    h, m, s = [int(x) for x in re.split(r"[.:]", t.strip())[:3]]
    return h * 3600 + m * 60 + s


def load_annotations(path):
    """Read a CAP annotation .txt file into a DataFrame.

    Returns columns: stage_col, position, clock, event, duration, location, onset_s
    onset_s = seconds since the first annotation row (handles passing midnight).
    """
    rows, header_seen = [], False
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if not header_seen:
                if line.lower().startswith("sleep stage"):
                    header_seen = True
                continue
            line = line.rstrip("\n")
            if not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) < 5:                       # fallback if spaces were used
                parts = re.split(r"\s{2,}", line.strip())
            parts = [p.strip() for p in parts] + [""] * (6 - len(parts))
            rows.append(parts[:6])

    df = pd.DataFrame(rows, columns=["stage_col", "position", "clock", "event", "duration", "location"])
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")

    # clock time -> seconds since start, fixing midnight rollover
    secs = df["clock"].map(_clock_to_sec).to_numpy()
    day_offset = np.cumsum(np.r_[0, (np.diff(secs) < 0).astype(int)]) * 86400
    secs = secs + day_offset
    df["onset_s"] = secs - secs[0]
    return df


ann = load_annotations(txt_path)
print(ann.shape)
ann.head(10)

In [ ]:
# What kinds of events are in this file?
ann["event"].value_counts()

### Split into sleep-stage epochs and CAP A-phases
- Stage rows have `event` starting with `SLEEP-`
- CAP A-phase rows have `event` starting with `MCAP-A`

In [ ]:
STAGE_MAP = {"S0": "W", "S1": "S1", "S2": "S2", "S3": "S3", "S4": "S4", "REM": "R", "MT": "MT"}
STAGE_ORDER = ["W", "R", "S1", "S2", "S3", "S4"]       # top -> bottom on the hypnogram
NREM = ["S1", "S2", "S3", "S4"]


def split_annotations(ann):
    is_stage = ann["event"].str.upper().str.startswith("SLEEP-")
    stages = ann[is_stage].copy()
    stages["stage"] = stages["event"].str.upper().str.replace("SLEEP-", "", regex=False).map(STAGE_MAP)
    stages = stages.reset_index(drop=True)

    is_cap = ann["event"].str.upper().str.startswith("MCAP-A")
    cap = ann[is_cap].copy()
    cap["subtype"] = cap["event"].str.upper().str[-2:]          # A1 / A2 / A3
    cap["end_s"] = cap["onset_s"] + cap["duration"]
    cap = cap.sort_values("onset_s").reset_index(drop=True)
    return stages, cap


stages, cap = split_annotations(ann)
print("Stage epochs:", len(stages), "| CAP A-phases:", len(cap))
stages.head()

## 6. Hypnogram

In [ ]:
def plot_hypnogram(stages, cap=None, title=""):
    y_map = {s: i for i, s in enumerate(reversed(STAGE_ORDER))}   # W at the top
    t_h = stages["onset_s"].to_numpy() / 3600
    y = stages["stage"].map(y_map).to_numpy()

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.step(t_h, y, where="post", lw=1, color="k")
    if cap is not None and len(cap):
        ax.vlines(cap["onset_s"] / 3600, -0.6, len(STAGE_ORDER) - 0.4,
                  color="tab:red", alpha=0.25, lw=0.6, label="CAP A-phase onset")
        ax.legend(loc="upper right")
    ax.set_yticks(list(y_map.values()))
    ax.set_yticklabels(list(y_map.keys()))
    ax.set_xlabel("Hours since start of annotation")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


plot_hypnogram(stages, cap, title=f"Hypnogram: {record}")

## 7. Basic sleep statistics

In [ ]:
def sleep_stats(stages):
    ep = EPOCH_SEC / 60                              # minutes per epoch
    counts = stages["stage"].value_counts()
    total_min = len(stages) * ep
    sleep_min = sum(counts.get(s, 0) for s in NREM + ["R"]) * ep
    out = {
        "recording_min": total_min,
        "TST_min": sleep_min,                                    # total sleep time
        "sleep_efficiency_%": 100 * sleep_min / total_min if total_min else np.nan,
        "NREM_min": sum(counts.get(s, 0) for s in NREM) * ep,
    }
    for s in ["W", "S1", "S2", "S3", "S4", "R"]:
        out[f"{s}_%_of_TST"] = 100 * counts.get(s, 0) * ep / sleep_min if (sleep_min and s != "W") else np.nan
    return pd.Series(out).round(1)


sleep_stats(stages)

In [ ]:
ax = stages["stage"].value_counts().reindex(STAGE_ORDER + ["MT"]).fillna(0).mul(EPOCH_SEC / 60).plot.bar(figsize=(6, 3))
ax.set_ylabel("Minutes")
ax.set_title(f"Time in each stage: {record}")
plt.tight_layout()
plt.show()

## 8. CAP A-phases and an (approximate) CAP rate

Each `MCAP-A1/A2/A3` row is an A-phase that belongs to a CAP sequence. We:
1. Count and measure A-phases by subtype
2. Group consecutive A-phases into **CAP sequences** (gap between A-phases of 60 s or less)
3. Approximate CAP time as the time spanned by each sequence, then divide by NREM time

> **Approximation warning:** the official CAP scoring rules (Terzano et al., 2001) also count the final B phase of a sequence.
> This estimate is close but not exact. For a paper, check the definition against the dataset's publication and the original CAP scoring rules.

In [ ]:
def cap_summary(stages, cap, max_gap_s=60):
    nrem_s = stages["stage"].isin(NREM).sum() * EPOCH_SEC
    if len(cap) == 0:
        return {"n_A": 0, "CAP_time_min": 0.0, "NREM_min": nrem_s / 60, "CAP_rate_%": 0.0}

    # group A-phases into sequences
    gap = cap["onset_s"].to_numpy()[1:] - cap["end_s"].to_numpy()[:-1]
    seq_id = np.r_[0, np.cumsum(gap > max_gap_s)]
    seq = cap.assign(seq=seq_id).groupby("seq").agg(start=("onset_s", "min"), end=("end_s", "max"),
                                                    n_A=("onset_s", "size"))
    seq = seq[seq["n_A"] >= 2]                            # a CAP sequence needs at least 2 A-phases
    cap_s = (seq["end"] - seq["start"]).sum()

    out = {
        "n_A": len(cap),
        "n_A1": (cap["subtype"] == "A1").sum(),
        "n_A2": (cap["subtype"] == "A2").sum(),
        "n_A3": (cap["subtype"] == "A3").sum(),
        "mean_A_dur_s": cap["duration"].mean(),
        "n_sequences": len(seq),
        "CAP_time_min": cap_s / 60,
        "NREM_min": nrem_s / 60,
        "CAP_rate_%": 100 * cap_s / nrem_s if nrem_s else np.nan,
    }
    return out


pd.Series(cap_summary(stages, cap)).round(2)

In [ ]:
# Distribution of A-phase durations by subtype
fig, ax = plt.subplots(figsize=(6, 3))
for sub, g in cap.groupby("subtype"):
    ax.hist(g["duration"], bins=range(0, 61, 2), alpha=0.5, label=sub)
ax.set_xlabel("A-phase duration (s)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Frequency content by sleep stage (optional)
Relative EEG band power per stage for the EEG channel picked earlier. You should see more **delta** in S3/S4 and more
**alpha** in wake.

This assumes the EDF and the annotation start at the same moment. Check `raw.info["meas_date"]` against the first clock time in the annotation file.

In [ ]:
BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 12), "sigma": (12, 16), "beta": (16, 30)}


def band_power_by_stage(raw, ch, stages):
    sf = raw.info["sfreq"]
    x = raw.get_data(picks=[ch])[0]
    rows = []
    for i, st in enumerate(stages["stage"]):
        a, b = int(i * EPOCH_SEC * sf), int((i + 1) * EPOCH_SEC * sf)
        if st == "MT" or b > len(x):
            continue
        f, p = welch(x[a:b], fs=sf, nperseg=int(4 * sf))
        total = p[(f >= 0.5) & (f <= 30)].sum()
        rows.append({"stage": st, **{k: p[(f >= lo) & (f < hi)].sum() / total for k, (lo, hi) in BANDS.items()}})
    return pd.DataFrame(rows).groupby("stage").mean().reindex(STAGE_ORDER)


bp = band_power_by_stage(raw, eeg_ch, stages)
bp.plot.bar(stacked=True, figsize=(7, 3.5))
plt.ylabel("Relative power")
plt.title(f"EEG band power by stage ({eeg_ch})")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 10. Compare CAP rate across groups (annotation files only)
The `.txt` files are tiny, so we can download the annotations for **all** recordings without touching the large EDF files.
This is a good first look at "CAP rate across disorder groups".

We try to read the official `RECORDS` list first and fall back to the expected file names if that fails.

In [ ]:
GROUP_NAMES = {
    "n": "Healthy", "nfle": "NFLE", "rbd": "RBD", "plm": "PLM",
    "ins": "Insomnia", "narco": "Narcolepsy", "sdb": "SDB", "brux": "Bruxism",
}


def get_record_names():
    try:
        p = download("RECORDS")
        names = [re.sub(r"\.edf$", "", l.strip()) for l in open(p) if l.strip()]
        names = [n for n in names if re.fullmatch(r"[a-z]+\d+", n)]
        if names:
            return sorted(set(names))
    except Exception as e:
        print("Could not read RECORDS, using fallback list:", e)
    counts = {"n": 16, "nfle": 40, "rbd": 22, "plm": 10, "ins": 9, "narco": 5, "sdb": 4, "brux": 2}
    return [f"{g}{i}" for g, n in counts.items() for i in range(1, n + 1)]


records = get_record_names()
print(len(records), "records, e.g.", records[:5])

In [ ]:
summaries = []
for rec in records:
    try:
        path = download(f"{rec}.txt")
        st, cp = split_annotations(load_annotations(path))
        s = cap_summary(st, cp)
        s.update(record=rec, group=GROUP_NAMES.get(re.match(r"[a-z]+", rec).group(), "?"),
                 TST_min=(st["stage"].isin(NREM + ["R"]).sum() * EPOCH_SEC / 60))
        summaries.append(s)
    except Exception as e:
        print(f"Skipping {rec}: {e}")

summary_df = pd.DataFrame(summaries).set_index("record")
summary_df.to_csv("cap_annotation_summary.csv")
summary_df.head()

In [ ]:
# CAP rate per group
order = summary_df.groupby("group")["CAP_rate_%"].median().sort_values().index
summary_df.groupby("group")["CAP_rate_%"].agg(["count", "mean", "std", "median"]).loc[order].round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
data = [summary_df.loc[summary_df["group"] == g, "CAP_rate_%"].dropna() for g in order]
ax.boxplot(data, showmeans=True)
ax.set_xticklabels(order)
ax.set_ylabel("CAP rate (%)  [approximate]")
ax.set_title("CAP rate by group")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 11. Things to keep in mind for modeling
- **Class imbalance:** groups differ a lot in size (e.g. NFLE has ~40 recordings while bruxism has ~2). Use subject-wise (not epoch-wise) train/test splits to avoid leakage.
- **Different channels and sampling rates** across recordings. Pick a channel set common to all subjects you use, and resample to one rate.
- **Medication and age** differ between groups and can affect CAP; check the PhysioNet page and the original paper for demographics.
- **Annotation quality:** A-phase labels come from human scoring; consider how that limits any model's ceiling.
- **Cite the dataset:** see the "Citation" section on the PhysioNet page (Terzano et al., 2001, plus the PhysioNet citation).

### Possible next steps
1. Extract features (band power, A-phase counts, CAP rate) per subject, then try a simple classifier (e.g. logistic regression / random forest) healthy vs one disorder.
2. Train a model to detect A-phases from raw EEG epochs.
3. Add per-subject metadata (age, sex, medication) to the summary table.